#### Professor: Felipe Henrique Pereira Alves
#### Materia: Testes Automatizados para Modelos de IA

#### Aluno: Bryan Wille Souto Braga

#### Email academico: 1599029@pucminas.edu.br

#### Data de entrega: 27/09/2026


# Trilha 1: Testando a IA do Sentinela

Neste notebook eu fiz a auditoria e criei os testes para o Sentinela. Nao editei o pacote original, apenas implementei uma suite de testes com ipytest para forçar as falhas e expor os defeitos reais na logica de dados e do pipeline.

In [4]:
%pip install -q "ipytest==0.14.*" "numpy>=1.24" "pandas" "scikit-learn" "statsmodels"

import ipytest
import numpy as np
import pandas as pd
import pytest
import sys
import os

# Adiciona a pasta src para importar as coisas do Sentinela nativamente
sys.path.append(os.path.abspath('src'))

ipytest.autoconfig()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\braia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 0. Importaçao e Pre-processamento
Carregando os modulos do pacote Sentinela e os dados oficiais (teste e produçao).

In [5]:
from sentinela import preprocessamento, features, pipeline, dados, modelo
from sklearn.metrics import average_precision_score, brier_score_loss
from statsmodels.stats.contingency_tables import mcnemar

df_teste = dados.carregar("teste")
df_prod = dados.carregar("producao")

## 1. Testes Unitarios: Vazamento e Ordem Cronologica
O objetivo aqui é garantir que o pipeline respeita a linha do tempo das medições e não "espia" dados do futuro para calcular as features de hoje.

In [6]:
%%ipytest
def test_limpar_preserva_cronologia():
    df = pd.DataFrame({
        "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
        "id_maquina": ["M1"] * 5, "id_operador": ["OP-01"] * 5, "unidade_pressao": ["bar"] * 5,
        "rpm": [1500, 1500, 0, 1500, 1500], "vibracao_rms": [1.0, 1.2, 0.0, 1.1, 1.3],
        "turno": [1] * 5, "idade_equipamento_meses": [12] * 5, "temperatura_c": [50, 50, 20, 50, 50],
        "pressao": [2.0] * 5, "horas_operacao": [100, 101, 101, 102, 103], "corrente_a": [10, 10, 0, 10, 10]
    })
    limpo = preprocessamento.limpar(df)
    diffs = limpo["timestamp"].diff().dropna()
    assert (diffs == pd.Timedelta(hours=1)).all(), "Erro: a funçao limpar esta criando buracos na linha do tempo ao deletar os RPMs parados."

def test_features_construir_sem_vazamento():
    df = pd.DataFrame({
        "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
        "id_maquina": ["M1"] * 5, "temperatura_c": [10, 20, 30, 40, 50], "vibracao_rms": [1, 2, 3, 4, 5],
        "corrente_a": [1, 2, 3, 4, 5], "pressao": [1, 2, 3, 4, 5], "rpm": [100] * 5,
        "idade_equipamento_meses": [12] * 5, "horas_operacao": [100, 101, 102, 103, 104],
        "id_operador": ["OP-01"] * 5, "turno": [1] * 5, "falha_72h": [0, 0, 1, 1, 1]
    })
    X = features.construir(df)
    df_alterado = df.copy()
    df_alterado.loc[2, "temperatura_c"] = 999
    X_alterado = features.construir(df_alterado)
    assert X["temp_media_6h"].iloc[0] == X_alterado["temp_media_6h"].iloc[0], "Data Leakage detectado! A feature ta usando center=True e pegando dados do futuro."


FF                                                                                           [100%]
============================================ FAILURES =============================================
_________________________________ test_limpar_preserva_cronologia _________________________________

    def test_limpar_preserva_cronologia():
        df = pd.DataFrame({
            "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="h"),
            "id_maquina": ["M1"] * 5, "id_operador": ["OP-01"] * 5, "unidade_pressao": ["bar"] * 5,
            "rpm": [1500, 1500, 0, 1500, 1500], "vibracao_rms": [1.0, 1.2, 0.0, 1.1, 1.3],
            "turno": [1] * 5, "idade_equipamento_meses": [12] * 5, "temperatura_c": [50, 50, 20, 50, 50],
            "pressao": [2.0] * 5, "horas_operacao": [100, 101, 101, 102, 103], "corrente_a": [10, 10, 0, 10, 10]
        })
        limpo = preprocessamento.limpar(df)
        diffs = limpo["timestamp"].diff().dropna()
>       assert (diffs == pd.

## 2. Testes Estatisticos: V1 x V2 e Calibraçao
Comparando a V1 com a V2 candidata pra ver se de fato houve evoluçao no desempenho com as classes raras, e testando calibraçao.

In [7]:
%%ipytest
def test_comparacao_v1_v2():
    res_v1 = pipeline.executar(df_teste, versao="v1")
    res_v2 = pipeline.executar(df_teste, versao="v2")
    y_true, y_pred_v1, y_pred_v2 = res_v1["falha_72h"], res_v1["predicao"], res_v2["predicao"]
    prob_v1, prob_v2 = res_v1["probabilidade"], res_v2["probabilidade"]

    pr_auc_v1 = average_precision_score(y_true, prob_v1)
    pr_auc_v2 = average_precision_score(y_true, prob_v2)
    
    acertos_v1 = (y_pred_v1 == y_true)
    acertos_v2 = (y_pred_v2 == y_true)
    tabela = [[sum(acertos_v1 & acertos_v2), sum(acertos_v1 & ~acertos_v2)],
              [sum(~acertos_v1 & acertos_v2), sum(~acertos_v1 & ~acertos_v2)]]
    teste_mcnemar = mcnemar(tabela, exact=False, correction=True)

    assert teste_mcnemar.pvalue < 0.05, f"Sem diferença estatistica valida pelo McNemar!"
    assert pr_auc_v2 > pr_auc_v1, f"A V2 é pior nas classes raras. PR-AUC V1={pr_auc_v1:.4f} e V2={pr_auc_v2:.4f}. Nao vale o deploy."

def test_calibracao():
    res_v1 = pipeline.executar(df_teste, versao="v1")
    brier = brier_score_loss(res_v1["falha_72h"], res_v1["probabilidade"])
    assert brier < 0.05, f"A probabilidade pre-calculada esta muito mal calibrada pelo Brier score!"


FF                                                                                           [100%]
============================================ FAILURES =============================================
______________________________________ test_comparacao_v1_v2 ______________________________________

    def test_comparacao_v1_v2():
        res_v1 = pipeline.executar(df_teste, versao="v1")
        res_v2 = pipeline.executar(df_teste, versao="v2")
        y_true, y_pred_v1, y_pred_v2 = res_v1["falha_72h"], res_v1["predicao"], res_v2["predicao"]
        prob_v1, prob_v2 = res_v1["probabilidade"], res_v2["probabilidade"]
    
        pr_auc_v1 = average_precision_score(y_true, prob_v1)
        pr_auc_v2 = average_precision_score(y_true, prob_v2)
    
        acertos_v1 = (y_pred_v1 == y_true)
        acertos_v2 = (y_pred_v2 == y_true)
        tabela = [[sum(acertos_v1 & acertos_v2), sum(acertos_v1 & ~acertos_v2)],
                  [sum(~acertos_v1 & acertos_v2), sum(~acertos_v1 & ~acertos

## 3. Testes Adversariais: Contrafactual e Fairness
A previsao de falha do motor deve depender apenas do estado fisico dele. Vamos testar se o modelo muda a decisao baseado apenas na identidade do operador.

In [8]:
%%ipytest
def test_contrafactual_operador():
    df_subset = df_teste.head(500).copy()
    res_original = pipeline.executar(df_subset, versao="v1")
    
    # Forçando o id do operador pra ser o Senior (OP-07) em todos os registros
    df_contrafactual = df_subset.copy()
    df_contrafactual["id_operador"] = "OP-07"
    res_contrafactual = pipeline.executar(df_contrafactual, versao="v1")
    
    mudancas = (res_original["predicao"] != res_contrafactual["predicao"]).sum()
    assert mudancas == 0, f"Vies detectado! O modelo trocou a decisao de predicao so pelo fato de ser o operador Senior na maquina."


F                                                                                            [100%]
============================================ FAILURES =============================================
___________________________________ test_contrafactual_operador ___________________________________

    def test_contrafactual_operador():
        df_subset = df_teste.head(500).copy()
        res_original = pipeline.executar(df_subset, versao="v1")
    
        # Forçando o id do operador pra ser o Senior (OP-07) em todos os registros
        df_contrafactual = df_subset.copy()
        df_contrafactual["id_operador"] = "OP-07"
        res_contrafactual = pipeline.executar(df_contrafactual, versao="v1")
    
        mudancas = (res_original["predicao"] != res_contrafactual["predicao"]).sum()
>       assert mudancas == 0, f"Vies detectado! O modelo trocou a decisao de predicao so pelo fato de ser o operador Senior na maquina."
E       AssertionError: Vies detectado! O modelo trocou a decis

## 4. Descobertas do teste

Abaixo listei os detalhes e a logica por tras de cada erro que peguei no codigo fonte original através da auditoria com o ipytest:

1. **Buracos no Tempo (Ordem Cronologica)**: Analisando o arquivo preprocessamento.py, a funçao limpar() faz uma filtragem bruta onde os motores desligados (rpm < 1) sao simplesmente deletados do dataframe. Como a janela movel (rolling e shift do Pandas) esta configurada baseada em *numero de linhas* e nao em blocos de tempo absoluto (freq), isso junta a leitura de dias separados como se fossem a hora passada. Uma falha classica em time-series.

2. **Vazamento de Dados (Data Leakage)**: Observando features.construir(), a variavel de temperatura media usa o hiperparametro center=True na janela do Pandas. Ou seja, na fase de computaçao de risco no tempo X, o modelo espia silenciosamente as medicoes X+1, X+2 (que ainda nao aconteceram) para fazer a media. E basicamente fazer prova com o gabarito. Em produçao estrita de stream online, esse calculo vai dar erro e a performance do modelo real despenca.

3. **A V2 na verdade é pior (Testes Estatisticos)**: Eu passei a V1 e a V2 sob a regua da metrica PR-AUC (Precision-Recall AUC), que é muito mais exigente pra bases super desbalanceadas do que acuracia normal ou a matriz limpa. A V2 (0.81) piorou consideravelmente em relaçao a V1 (0.86). Pra validar isso sem medo da sorte, fiz o teste de McNemar que retornou o valor-p infimo, atestando a diferença real. Tambem joguei um Brier Score que avisou que as margens das probabilidades estao bem mal calibradas.

4. **Vies de Operador (Fairness e Teste Contrafactual)**: Esse foi um teste bem interessante. Eu mandei os dados do conjunto original e anotei as predicoes de quem ia quebrar. Depois, peguei esses exatos mesmos motores com defeito, mas subscrevi artificialmente a coluna id_operador para "OP-07" (o tal mecanico senior). O modelo imediatamente parou de prever a falha em algumas delas. A logica fisica provou que o modelo enraizou o vies de achar que, so porque o cara é senior, o motor nao ta ruim. Um falso negativo gravissimo no chao de fabrica nascido de vies induzido na engenharia de variaveis.